In [ ]:
import pandas as pd
from time import sleep

from pmcortex.mesh import extract_mesh_coarsest_descriptor_categories
from pmcortex.pmcsearch import PMCSearch

# define a set of MeSH categories to use as search queries, based on the coarsest descriptor categories in the MeSH hierarchy
cats = extract_mesh_coarsest_descriptor_categories("../data/desc2026.xml", level=2)
use_keywords = {
    c["tree_number"]: c["name"]
    for c in cats
    if c["tree_number"][0] in ['A', 'B', 'C', 'D', 'E', 'G']
}

# use the MeSH categories as search queries to get seed PMC articles
seed_pmcs = []
serach = PMCSearch()
for mesh, query in use_keywords.items():
    serach.search(query)
    df_ = serach.get_all()
    df_ = df_.assign(query = query)
    df_ = df_.assign(mesh = mesh)
    seed_pmcs.append(df_)
    sleep(1.1)

# combine and filter the seed PMC articles
seed_pmcs = pd.concat(seed_pmcs)
filtered_seeds = (
    seed_pmcs
    .query('not title.str.contains("correction", case=False)')
    .query('pmcid.str.startswith("PMC")')
    .groupby(['pmcid', 'title'])
    .agg(list)
    .reset_index()
)
filtered_seeds.to_csv("../data/mesh_based_pmc_seed.csv", index=False)

KeyboardInterrupt: 

In [ ]:
import pandas as pd
from pmcortex.downloader import PMCJATSDownloader

filtered_seeds = pd.read_csv("../data/mesh_based_pmc_seed.csv")
with PMCJATSDownloader(
    out_dir="../data/pmc_context_queries_v0-5/xml_jats",
    user_agent="PMCortex/0.2 (contact: lorenz@hexemer.eu)",
    max_retries=3,
) as dl:
    results = dl.download_many(filtered_seeds['pmcid'].unique())

In [ ]:
import os
total_size = 0
counter = 0
for file in os.scandir('../data/pmc_context_queries_v0-5/xml_jats'):
    if file.is_file() and file.name.endswith('.nxml'):
        file_info = file.stat()
        total_size += file_info.st_size
        counter += 1
print(f"Total size: {total_size / (1024 * 1024):.2f} MB")
print(f"Number of files: {counter}")
print(f"Average size per file: {total_size / counter / 1024:.2f} KB")

Total size: 1489.80 MB
Number of files: 10834
Average size per file: 140.81 KB


In [ ]:
import os
import pandas as pd

filtered_seeds = pd.read_csv("../data/mesh_based_pmc_seed.csv").set_index('pmcid')
filtered_seeds = filtered_seeds.assign(loaded = False)
for file in os.scandir('../data/pmc_context_queries_v0-5/xml_jats'):
    if file.is_file() and file.name.endswith('.nxml'):
        pmcid = file.name.replace('.nxml', '')
        filtered_seeds.loc[pmcid, 'loaded'] = True

In [1]:
from pathlib import Path
import yaml
import os
import tqdm

from pmcortex.jatsparser import JATSParser

abstracts = Path('../data/pmc_context_queries_v0-5/extracted_metadata.csv')
abstracts.unlink(missing_ok=True)
contexts = Path('../data/pmc_context_queries_v0-5/extracted_contexts.csv')
contexts.unlink(missing_ok=True)
sources = Path('../data/pmc_context_queries_v0-5/extracted_sources.csv')
sources.unlink(missing_ok=True)
text_path = Path('../data/pmc_context_queries_v0-5/fulltexts/')
text_path.mkdir(exist_ok=True, parents=True)
author_path = Path('../data/pmc_context_queries_v0-5/authors/')
author_path.mkdir(exist_ok=True, parents=True)



for file in tqdm.tqdm(os.scandir("../data/pmc_context_queries_v0-5/xml_jats")):
    
    if file.name.endswith(".nxml"):
        parser = JATSParser()
        article = parser.parse_article(file.path, save_sentences=True)
        
        sources_df = parser.sources_to_dataframe()
        sources_df = sources_df.assign(source_pmcid=article.pmcid)
        sources_df.to_csv(
            str(sources),
            mode='a' if sources.exists() else 'w',
            header=not sources.exists(),
            index=None
        )

        context_df = parser.contexts_to_dataframe()
        context_df = context_df.assign(source_pmcid=article.pmcid)
        context_df.to_csv(
            str(contexts),
            mode='a' if contexts.exists() else 'w',
            header=not contexts.exists(),
            index=None
        )

        abstract_df = parser.metadata_to_dataframe()
        abstract_df.to_csv(
            str(abstracts),
            mode='a' if abstracts.exists() else 'w',
            header=not abstracts.exists(),
            index=None
        )

        fulltext = parser.text_to_list()
        with open(text_path / file.name.replace('.nxml', '.txt'), 'w') as fp:
            fp.writelines(fulltext)


        authors_and_refs = parser.normalized_authors_and_refs()
        with open(author_path / file.name.replace('.nxml', ".yaml"), 'w', encoding="utf-8") as fp:
            fp.write(
                yaml.safe_dump(authors_and_refs, sort_keys=False, allow_unicode=False),
            )   

10834it [04:55, 36.70it/s]


In [2]:
import os
total_size = 0
counter = 0
for file in os.scandir('../data/pmc_context_queries_v0-5/fulltexts'):
    if file.is_file() and file.name.endswith('.txt'):
        file_info = file.stat()
        total_size += file_info.st_size
        counter += 1
print(f"Total size: {total_size / (1024 * 1024):.2f} MB")
print(f"Number of files: {counter}")
print(f"Average size per file: {total_size / counter / 1024:.2f} KB")

Total size: 326.02 MB
Number of files: 10834
Average size per file: 30.81 KB


In [16]:
import os
total_size = 0
counter = 0
for file in os.scandir('../data/pmc_context_queries_v0-5/authors'):
    if file.is_file() and file.name.endswith('.yaml'):
        file_info = file.stat()
        total_size += file_info.st_size
        counter += 1
print(f"Total size: {total_size / (1024 * 1024):.2f} MB")
print(f"Number of files: {counter}")
print(f"Average size per file: {total_size / counter / 1024:.2f} KB")

Total size: 29.86 MB
Number of files: 10834
Average size per file: 2.82 KB


In [1]:
from pmcortex.query_filters import QueryFilterPipeline

pipeline = QueryFilterPipeline("../data/pmc_context_queries_v0-5")
print(f"Remaining rows: {len(pipeline.filtered_df)}")

# base filters
pipeline.filter_no_anaphoric_starts()
pipeline.filter_no_reference_framing_starts()
# pipeline.filter_no_summary_starts()
pipeline.filter_max_strangeness(0.3)
print(f"Remaining rows {len(pipeline.filtered_df)} after base filters")

# refference cound based filters (defines benshmark)
pipeline.filter_min_hit_queries(1)
pipeline.filter_min_source_count(20)
print(f"Remaining rows {len(pipeline.filtered_df)} after hit and source count filters")

# remove spoilers and sel-fcitations
pipeline.filter_no_et_al_queries()
pipeline.filter_no_explicit_spoilers()
pipeline.filter_no_self_citations()
print(f"Remaining rows {len(pipeline.filtered_df)} after spoiler and self-citation filters")

pipeline.cleanup_queries()
pipeline.filter_min_query_length(6)
pipeline.filter_max_query_length(100)
print(f"Remaining rows {len(pipeline.filtered_df)} after cleanup and length filters")

df_queries = pipeline.filtered_df[['query', 'context', 'source_pmcid', 'hits_parsed', 'n_sources']].rename(columns={'hits_parsed': 'hits'})

Remaining rows: 421858
Remaining rows 387320 after base filters
Remaining rows 231744 after hit and source count filters
Remaining rows 171843 after spoiler and self-citation filters
Remaining rows 170430 after cleanup and length filters


In [7]:
relevant_pmcids = pipeline.filtered_df.value_counts('source_pmcid').reset_index().query('1 <= count <= 2')['source_pmcid'].tolist()
selected_df, selected_refs = pipeline.select_shared_reference_subset(n=19, relevant_pmcids=relevant_pmcids)
len(selected_df), len(selected_refs)

(318, 3593)

In [ ]:
selected_df[['source_pmcid', 'context', 'query', 'hits_parsed', 'selected_decoys']].to_csv("../data/pmc_context_queries_v0-5/benchmark_queries_1-2.csv", index=False)

In [2]:
import tqdm
import pandas as pd
from pathlib import Path
from pmcortex import PubMedMeSHClient, extract_abstracts_from_pubmed_xml, extract_mesh_terms_from_pubmed_xml

ABSTACT_DB = Path("../data/pmc_context_queries_v0-5/pmid_abstracts.csv")
MESH_DB = Path("../data/pmc_context_queries_v0-5/pmid_mesh_terms.csv")
ABSTACT_DB.unlink(missing_ok=True)
MESH_DB.unlink(missing_ok=True)

selected_df = pd.read_csv("../data/pmc_context_queries_v0-5/benchmark_queries_1-2.csv")
required_pmids = (
    set(selected_df['selected_decoys'].apply(eval).explode())
    .union(set(selected_df['hits_parsed'].apply(eval).explode()))
)


with PubMedMeSHClient() as client:
    for i in tqdm.tqdm(range(0, len(required_pmids), 1000), desc="Fetching PubMed metadata"):
        batch_pmids = list(required_pmids)[i:i+1000]
        xml = client.fetch_pubmed_xml(batch_pmids)
        abstracts = extract_abstracts_from_pubmed_xml(xml)
        mesh_terms = extract_mesh_terms_from_pubmed_xml(xml)
        
        abstracts.to_csv(ABSTACT_DB, mode='a', header=not ABSTACT_DB.exists(), index=False)
        mesh_terms.to_csv(MESH_DB, mode='a', header=not MESH_DB.exists(), index=False)

Fetching PubMed metadata: 100%|██████████| 16/16 [01:19<00:00,  4.95s/it]


In [92]:
import pandas as pd
from pathlib import Path

MESH_DB = Path("../data/pmc_context_queries_v0-5/pmid_mesh_terms.csv")
QUERY_DB = Path("../data/pmc_context_queries_v0-5/benchmark_queries_1-2.csv")

all_mesh = pd.read_csv(MESH_DB)
all_mesh['pmid'] =  all_mesh['pmid'].astype(str)
selected_df = pd.read_csv(QUERY_DB)
selected_df['selected_decoys'] = selected_df['selected_decoys'].apply(eval)
selected_df['hits_parsed'] = selected_df['hits_parsed'].apply(lambda x: eval(x)[0])

In [142]:
selected_df['soft_decoys'] = None
selected_df['selected_decoy_similarities'] = None
selected_df['soft_decoy_similarities'] = None
selected_df['n_hit_meshs'] = None

for i, row in tqdm.tqdm(selected_df.iterrows(), total=len(selected_df), desc="Calculating MeSH similarities"):
    pmcid = row['source_pmcid']
    hit_pmid = row['hits_parsed']
    other_pmcids = selected_df.query('source_pmcid != @pmcid')
    other_pmids = other_pmcids.sample(5)['selected_decoys'].explode().sample(80)

    hit_meshs = all_mesh.query('pmid == @hit_pmid')['descriptor_name'].to_list()
    if len(hit_meshs) == 0:
        continue

    decoy_meshs = [all_mesh.query('pmid == @decoy')['descriptor_name'].to_list() for decoy in row['selected_decoys']]
    soft_decoy_meshs = [all_mesh.query('pmid == @decoy')['descriptor_name'].to_list() for decoy in other_pmids]
    
    decoy_mesh_similarities = [len(set(d_mesh).intersection(set(hit_meshs)))/len(hit_meshs) for d_mesh in decoy_meshs]
    soft_decoy_mesh_similarities = [len(set(d_mesh).intersection(set(hit_meshs)))/len(hit_meshs)for d_mesh in soft_decoy_meshs]
    row['selected_decoy_similarities'] = decoy_mesh_similarities
    row['soft_decoy_similarities'] = soft_decoy_mesh_similarities
    row['soft_decoys'] = other_pmids.tolist()
    row['n_hit_meshs'] = len(hit_meshs)
    selected_df.loc[i] = row

Calculating MeSH similarities: 100%|██████████| 2858/2858 [12:58<00:00,  3.67it/s]


In [144]:
selected_df.to_csv("../data/pmc_context_queries_v0-5/benchmark_queries_1-2_softdecoys.csv", index=False)

In [137]:
selected_df

,source_pmcid,context,query,hits_parsed,selected_decoys,soft_decoys,selected_decoy_similarities,soft_decoy_similarities
0,PMC9967420,An overexposure of more than three mm is known...,"In some European countries, a gingival display...",3163980,"[10863370, 20516094, 22368373, 17586346, 21845...","[17210947, 21687657, 16740312, 17125278, 27698...","[0.2, 0.5, 0.0, 0.2, 0.4, 0.1, 0.0, 0.0, 0.1, ...","[0.1, 0.1, 0.1, 0.1, 0.0, 0.1, 0.1, 0.1, 0.0, ..."
1,PMC9967420,"In some European countries, a gingival display...",Etiological factors related to a gummy smile c...,6583388,"[10863370, 20516094, 22368373, 17586346, 21845...","[23871583, 26637207, 16107889, 25722581, 21367...","[0.1, 0.3, 0.0, 0.1, 0.6, 0.1, 0.0, 0.1, 0.2, ...","[0.3, 0.1, 0.4, 0.0, 0.4, 0.1, 0.0, 0.4, 0.4, ..."
2,PMC9967420,Other studies have shown the importance of tre...,It can lead to certain complications (mainly p...,28000276,"[10863370, 20516094, 22368373, 17586346, 21845...","[15894917, 16371768, 16965158, 25565766, 26899...","[0.14285714285714285, 0.2857142857142857, 0.0,...","[0.14285714285714285, 0.14285714285714285, 0.0..."
3,PMC9967420,"According to de Brandao et al., there is no ev...",Several studies demonstrated a higher success ...,28000276,"[10863370, 20516094, 22368373, 17586346, 21845...","[16136525, 12432537, 15976586, 1527693, 225776...","[0.14285714285714285, 0.2857142857142857, 0.0,...","[0.0, 0.0, 0.14285714285714285, 0.142857142857..."
4,PMC9972740,Testicular hypoplasia can affect the sexual an...,It is very important to study the influencing ...,35193821,"[29183799, 27642851, 25615240, 15499792, 20450...","[20466091, 15355426, 15021220, 20506187, 19390...","[0.42857142857142855, 0.5714285714285714, 0.57...","[0.14285714285714285, 0.14285714285714285, 0.1..."
...,...,...,...,...,...,...,...,...
2853,PMC9964436,These methods include the production of formal...,The possible formation of various hydroxylatio...,25840124,"[7023838, 90918, 30236787, 6323433, 5466067, 3...",None,None,None
2854,PMC9964436,"Nonetheless, the possible formation of various...",There are a number of assays which take advant...,6278984,"[7023838, 90918, 30236787, 6323433, 5466067, 3...",None,None,None
2855,PMC9964436,"Thus, there are a number of assays which take ...",In the case of the salicylate hydroxylation as...,25840124,"[7023838, 90918, 30236787, 6323433, 5466067, 3...",None,None,None
2856,PMC9964436,For instance in the case of the salicylate hyd...,A further problem with salicylate is that it h...,16244735,"[7023838, 90918, 30236787, 6323433, 5466067, 3...",None,None,None


In [8]:
relevant_pmcids = pipeline.filtered_df.value_counts('source_pmcid').reset_index().query('2 <= count <= 3')['source_pmcid'].tolist()
selected_df, selected_refs = pipeline.select_shared_reference_subset(n=19, relevant_pmcids=relevant_pmcids)
len(selected_df), len(selected_refs)

(823, 5977)

In [10]:
relevant_pmcids = pipeline.filtered_df.value_counts('source_pmcid').reset_index().query('3 <= count <= 5')['source_pmcid'].tolist()
selected_df, selected_refs = pipeline.select_shared_reference_subset(n=19, relevant_pmcids=relevant_pmcids)
len(selected_df), len(selected_refs)

(2858, 12965)

In [23]:
relevant_pmcids = pipeline.filtered_df.value_counts('source_pmcid').reset_index().query('7 < count < 10')['source_pmcid'].tolist()
len(pipeline.filtered_df.query('source_pmcid in @relevant_pmcids'))

6169

In [24]:
relevant_pmcids = pipeline.filtered_df.value_counts('source_pmcid').reset_index().query('8 < count < 12')['source_pmcid'].tolist()
len(pipeline.filtered_df.query('source_pmcid in @relevant_pmcids'))

10505

# Remove quieries which depend on refferences to the context

In [3]:
import pandas as pd
from tqdm import tqdm
from pmcortex.query_filters import DEFAULT_ANAPHORIC_STARTS

from IPython.display import clear_output, display

In [25]:
from fastcoref import spacy_component
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading language model for the spaCy pipeline because it isn't installed")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

nlp.add_pipe(
   "fastcoref", 
   #config={'model_architecture': 'LingMessCoref', 'model_path': 'biu-nlp/lingmess-coref', 'device': 'cpu'}
)


06/10/2026 00:24:19 - INFO - 	 missing_keys: []
06/10/2026 00:24:19 - INFO - 	 unexpected_keys: []
06/10/2026 00:24:19 - INFO - 	 mismatched_keys: []
06/10/2026 00:24:19 - INFO - 	 error_msgs: []
06/10/2026 00:24:19 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M


In [5]:
selected_df = pd.read_csv("../data/pmc_context_queries_v0-5/benchmark_queries_1-2_softdecoys.csv")

In [22]:
coref_clusters = []
texts_to_process = []
indices_to_process = []

# Prepare all texts first
for i, row in selected_df.query('not context.isnull()').iterrows():
    full_text = row['context'] + " " + row['query']
    texts_to_process.append(full_text)
    indices_to_process.append((i, row, len(row['context'])))

# Process all texts at once with pipe
docs = list(nlp.pipe(texts_to_process, batch_size=512))

# Now process the results
for doc, (i, row, context_length) in zip(docs, indices_to_process):
    if not doc._.coref_clusters:
        continue

    full_text = texts_to_process[indices_to_process.index((i, row, context_length))]
    
    for cluster in doc._.coref_clusters:
        word_ends = [b for _, b in cluster]
        if max(word_ends) > context_length and min(word_ends) < context_length:
            cluster_words_context = [full_text[a:b] for a, b in cluster if b <= context_length]
            cluster_words_query = [full_text[a:b] for a, b in cluster if b > context_length]
            query_markers = [
                marker
                for marker in DEFAULT_ANAPHORIC_STARTS
                if any(marker == word.lower() for word in cluster_words_query)
            ]
            if query_markers:
                coref_clusters.append({
                    'index': i,
                    'full_text': full_text,
                    'cluster_words_context': cluster_words_context,
                    'cluster_words_query': cluster_words_query,
                    'query_markers': query_markers
                })

coref_clusters = pd.DataFrame(coref_clusters)

06/10/2026 00:21:07 - INFO - 	 Tokenize 512 inputs...
Map: 100%|██████████| 512/512 [00:00<00:00, 734.25 examples/s]
06/10/2026 00:21:07 - INFO - 	 ***** Running Inference on 512 texts *****
Inference: 100%|██████████| 512/512 [00:01<00:00, 495.82it/s]
06/10/2026 00:21:10 - INFO - 	 Tokenize 512 inputs...
Map: 100%|██████████| 512/512 [00:00<00:00, 705.30 examples/s]
06/10/2026 00:21:11 - INFO - 	 ***** Running Inference on 512 texts *****
Inference: 100%|██████████| 512/512 [00:00<00:00, 620.24it/s]
06/10/2026 00:21:14 - INFO - 	 Tokenize 512 inputs...
Map: 100%|██████████| 512/512 [00:00<00:00, 661.91 examples/s]
06/10/2026 00:21:15 - INFO - 	 ***** Running Inference on 512 texts *****
Inference: 100%|██████████| 512/512 [00:00<00:00, 584.60it/s]
06/10/2026 00:21:18 - INFO - 	 Tokenize 512 inputs...
Map: 100%|██████████| 512/512 [00:00<00:00, 670.60 examples/s]
06/10/2026 00:21:19 - INFO - 	 ***** Running Inference on 512 texts *****
Inference: 100%|██████████| 512/512 [00:00<00:00, 

In [20]:
coref_clusters.to_csv('../data/pmc_context_queries_v0-5/benchmark_queries_1-2_coref_clusters.csv', index=False)

In [14]:
coref_clusters[coref_clusters['cluster_words_query'].apply(lambda x: len(x) > 1)]

,index,full_text,cluster_words_context,cluster_words_query,query_markers
4,135,Several studies have shown that using ultrasou...,[ultrasound (US) guidance],"[US, it]",[it]
8,215,"According to a previous analysis, patients wit...","[second-generation H1-antihistamines, they]","[Its, it, its]","[it, its]"
10,287,"Currently, ribavirin analogues with substituen...",[ribavirin analogues with substituents on the ...,"[Thio derivatives of 1,2,4-triazoles, their]",[their]
11,291,"In this state, these digestive enzymes erode b...",[these digestive enzymes],"[digestive enzymes, these enzymes, their, thei...",[their]
12,333,"The presence of M. morganii, which has demonst...","[M. morganii, which has demonstrated high degr...","[its, Dysgonomonas, the microbial consortia’s]",[its]
18,499,Hexafluoroisopropanol (HFIP) and trifluoroethy...,[Hexafluoroisopropanol (HFIP)],"[HFIP, its]",[its]
21,557,This cohort is comprised of a different set of...,[participants who were enrolled after publicat...,"[the participants’, their]",[their]
22,560,This technological innovation not only augment...,[This technological innovation],"[this technique, their, it, their]","[it, their]"
23,581,"By minimizing movement, these pins negate the ...",[these pins],"[their, their]",[their]
25,608,It is usually incidentally found in the second...,[It],"[It, it]",[it]


In [17]:
coref_clusters.iloc[75].full_text

'Most of these utilize the general-purpose RNA-cleaving deoxyribozymes 8–17 and 10–23 isolated by Santoro and Joyce. Under the optimal conditions, 10–23 cleaves phosphodiester bonds with a catalytic efficiency of 109 M−1min−1, making it the fastest nucleic acid enzyme known.'